# Assignment 2 – Live Test in Colab
**Workspace Assistant (Google ADK): Google Calendar tools + GitHub MCP + tool search bonus**

This notebook runs the real agent from `MyTeachers123/agents-assignment-2` against **your real Google Calendar and GitHub**, and prints every tool call so you can see it working.

**Before you start – Colab secrets (🔑 icon on the left, turn *Notebook access* ON):**
| Secret name | Value |
|---|---|
| `GOOGLE_API_KEY` | Gemini API key (same as the Week 2 lab) |
| `GITHUB_PERSONAL_ACCESS_TOKEN` | your `ghp_...` token |

You also need the `credentials.json` file (Google OAuth *Desktop app* client) on your computer.
Everything below runs on Colab's temporary machine, not on your computer.

**How to run:** run the cells **one by one from the top**. Cell 1 restarts Colab automatically at the end (you will see *"Your session crashed"* – that is expected). Then continue with cell 2.

*Saved here as evidence of the live test; outputs are cleared to keep calendar data private.*

## 1. Get the code, install, and restart (run once)

In [ ]:
import os
if not os.path.exists("/content/agents-assignment-2"):
    !git clone -q https://github.com/MyTeachers123/agents-assignment-2.git
%cd /content/agents-assignment-2/workspace_assistant
# -U upgrades Colab's pre-installed (older) mcp / google-adk
!pip install -q -U -r requirements.txt "mcp>=2" "google-adk>=2.11"
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs npm && node --version)
print("\n✅ Installed. Colab will now RESTART automatically (a 'session crashed' message is expected).")
print("➡️  After it restarts, continue with cell 2. Do NOT run cell 1 again.")
import time, sys; sys.stdout.flush(); time.sleep(3)
os.kill(os.getpid(), 9)   # restart Python so the upgraded packages are used

## 2. Load secrets and settings

In [ ]:
%cd /content/agents-assignment-2/workspace_assistant
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
os.environ["GITHUB_PERSONAL_ACCESS_TOKEN"] = userdata.get("GITHUB_PERSONAL_ACCESS_TOKEN")
os.environ["GITHUB_USERNAME"] = "MyTeachers123"
os.environ["MODEL_NAME"] = "gemini-3.8-flash"
os.environ["CALENDAR_TIMEZONE"] = "America/Los_Angeles"
print("✅ Secrets loaded (values hidden)")

import importlib.metadata as md
from google.adk.tools.mcp_tool import McpToolset   # fails here if cell 1's restart did not happen
print("✅ mcp", md.version("mcp"), "| google-adk", md.version("google-adk"))

## 3. Upload `credentials.json` (Google OAuth Desktop client)

In [ ]:
from google.colab import files
from pathlib import Path

cred_file = Path("config/credentials/credentials.json")
if cred_file.exists():
    print("✅ credentials.json already uploaded – skipping")
else:
    up = files.upload()                   # choose credentials.json
    cred_file.write_bytes(up[next(iter(up))])
    print("✅ Saved", cred_file)

## 4. Sign in to Google Calendar (one time)
1. Run the cell and click the printed link. Sign in with the Gmail you added as a **Test user**.
2. If you see *Google hasn't verified this app* → **Continue**. Allow calendar access.
3. The browser then goes to a page that **fails to load** (`http://localhost/?state=...&code=...`). That is expected.
4. **Copy that whole address from the address bar** and paste it into the box below.

In [ ]:
from urllib.parse import urlparse, parse_qs
from google_auth_oauthlib.flow import InstalledAppFlow
from tools.auth import SCOPES, CREDENTIALS_FILE, TOKEN_FILE

if TOKEN_FILE.exists():
    print("✅ Already signed in (token.json found) – skipping")
else:
  flow = InstalledAppFlow.from_client_secrets_file(str(CREDENTIALS_FILE), SCOPES["calendar"],
                                                   redirect_uri="http://localhost")
  auth_url, _ = flow.authorization_url(prompt="consent", access_type="offline")
  print("👉 Open this link:\n", auth_url)

  redirected = input("\nPaste the full http://localhost/... address here: ").strip()
  code = parse_qs(urlparse(redirected).query).get("code", [redirected])[0]
  flow.fetch_token(code=code)
  TOKEN_FILE.write_text(flow.credentials.to_json())
  print("✅ Signed in. token.json saved on this Colab machine only.")

## 5. Call the calendar tools directly (no LLM) – proves the Google API works

In [ ]:
from tools.calendar_tools import get_current_datetime, list_upcoming_events
from datetime import date, timedelta

now = get_current_datetime()          # date in Los Angeles (Colab's clock is UTC)
print(now)
today = date.fromisoformat(now["date"])
week = list_upcoming_events(start_date=str(today), end_date=str(today + timedelta(days=7)))
print("status:", week["status"], "| events in the next 7 days:", week.get("count"))
for e in week.get("events", [])[:5]:
    print(" -", e["start"], "|", e["summary"])

## 6. Build the agent + a `chat()` helper that shows every tool call
`mode="full"` loads all GitHub tools; `mode="search"` uses the tool-search bonus (defer_loading).

In [ ]:
import json, uuid
from google.adk.runners import InMemoryRunner
from google.genai import types
from agent import create_agent, create_agent_with_tool_search

runners, sessions = {}, {}

def _runner(mode):
    if mode not in runners:
        agent = create_agent_with_tool_search() if mode == "search" else create_agent()
        r = InMemoryRunner(agent=agent, app_name=f"wa_{mode}")
        r.auto_create_session = True
        runners[mode], sessions[mode] = r, uuid.uuid4().hex
    return runners[mode]

def _short(v, n=110):
    s = json.dumps(v, default=str, ensure_ascii=False)
    return s if len(s) <= n else s[:n] + "…"

async def _chat_once(text, mode="full"):
    print(f"🧑 {text}")
    runner, tokens = _runner(mode), []
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    async for ev in runner.run_async(user_id="colab", session_id=sessions[mode], new_message=msg):
        u = getattr(ev, "usage_metadata", None)
        if u and u.prompt_token_count:
            tokens.append(u.prompt_token_count)
        for p in (ev.content.parts if ev.content and ev.content.parts else []):
            if p.function_call:
                print(f"   🔧 {p.function_call.name}({_short(dict(p.function_call.args or {}))})")
            elif p.function_response:
                r = p.function_response.response or {}
                status = r.get("status") or ("error" if r.get("isError") else "ok") if isinstance(r, dict) else "ok"
                extra = f" | {r.get('message')}" if isinstance(r, dict) and r.get("message") else ""
                print(f"   ↳ {p.function_response.name} → {status}{extra}")
            elif p.text and not getattr(p, "thought", False):
                print(f"🤖 {p.text.strip()}")
    print(f"   📊 prompt tokens per model call: {tokens}  (mode={mode})\n")

import asyncio, logging
logging.getLogger("google_adk").setLevel(logging.CRITICAL)   # hide long tracebacks

async def chat(text, mode="full", tries=4):
    """Like _chat_once, but waits and retries when Gemini is busy (503) or rate-limited (429)."""
    for i in range(tries):
        try:
            return await _chat_once(text, mode)
        except Exception as e:
            busy = any(s in str(e) for s in ("503", "UNAVAILABLE", "429", "RESOURCE_EXHAUSTED"))
            if busy and i < tries - 1:
                wait = 20 * (i + 1)
                print(f"   ⏳ Gemini is busy, retrying in {wait}s…")
                await asyncio.sleep(wait)
            else:
                print(f"   ❌ {type(e).__name__}: {str(e)[:200]}")
                return

def new_chat(mode="full"):
    sessions[mode] = uuid.uuid4().hex
print("✅ Ready")

## 7. Calendar tests

In [ ]:
await chat("What meetings do I have tomorrow?")

In [ ]:
await chat("Find a free 1-hour slot next week")

In [ ]:
await chat("Am I free tomorrow at 3pm?")

Create a test event. The agent should **ask for confirmation first**; the next cell answers *yes*.

In [ ]:
await chat('Schedule "Assignment 2 test" tomorrow at 3pm for 30 minutes')

In [ ]:
await chat("Yes, go ahead")

In [ ]:
await chat('Move "Assignment 2 test" to 4pm')

In [ ]:
await chat("Yes")

Error handling check (should explain the problem, not crash):

In [ ]:
await chat("Schedule a meeting on 2020-01-01 at 10am")

## 8. GitHub (MCP) tests

In [ ]:
await chat("List my GitHub repositories")

In [ ]:
await chat("Show open issues in octocat/Hello-World")

In [ ]:
await chat("Show me the README from octocat/Hello-World")

## 9. Bonus: tool search (defer_loading)
Same questions in `search` mode. Watch for `search_github_tools` being called first, and compare the prompt tokens with section 8.

In [ ]:
await chat("List my GitHub repositories", mode="search")

In [ ]:
await chat("Show me the README from octocat/Hello-World", mode="search")

Side-by-side numbers for the reflection (real Gemini prompt tokens):

In [ ]:
import scripts.compare_context_tokens as cmp
await cmp.main()

## 10. Clean up the test event (optional)

In [ ]:
from tools.auth import get_calendar_service
svc = get_calendar_service()
found = list_upcoming_events(search_text="Assignment 2 test", max_results=5)
for e in found.get("events", []):
    svc.events().delete(calendarId="primary", eventId=e["event_id"]).execute()
    print("🗑️ deleted:", e["summary"], e["start"])
print("done")

---
⚠️ **Privacy:** the outputs above show your real calendar events. If you save this notebook to a **public** repo, use *Edit → Clear all outputs* first, or keep only test events visible.